# AmbiFix: NEW original-data square-root-weighted BERT run

**RTX 5060 Ti 16GB / Windows / Run All.** This retrains a new classifier from the pinned original `bert-base-uncased` checkpoint. The historical validation Macro F1 **0.581140 is a reference, not a guaranteed score or recovered model**.

Training: only the packaged frozen 4,731 original training rows. Selection: only the packaged frozen 1,015 validation rows. No augmented data or final held-out input/entry point is included. Every result is a new run with its own complete checkpoint, predictions and actual logs.

Read `README_WINDOWS.md` and `HYPERPARAMETER_PROVENANCE.md` before execution. Extract the entire ZIP, keep its folders together, create the documented Windows Python environment, open this notebook, and choose **Run → Run All Cells**. The notebook performs input checks, prepares pinned dependencies, then executes exactly one CUDA/BF16 training run. It will not train on a Mac/CPU/MPS or automatically retry an interrupted run. At least **12GiB free disk** is required for complete transactional checkpoints/return files.


In [ ]:
# Locate the extracted package without walking any dataset directory.
from pathlib import Path
import json, os, sys, subprocess, platform
candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
roots = [p for p in candidates if (p / "data_manifest.json").is_file() and (p / "scripts/train_bert.py").is_file()]
if len(roots) != 1:
    raise RuntimeError("Open Jupyter from the extracted package folder (or its notebooks folder), then Run All again. No files were modified.")
PACKAGE_ROOT = roots[0]
sys.path.insert(0, str(PACKAGE_ROOT / "scripts"))
from verify_inputs import verify_inputs
CHECKED = verify_inputs(PACKAGE_ROOT)
print("New experiment:", CHECKED["config"]["experiment_id"])
print("Frozen original training rows:", len(CHECKED["train_rows"]))
print("Frozen validation rows:", len(CHECKED["validation_rows"]))
print("Training-only square-root class weights:", CHECKED["class_weights"])
print("Input preflight passed. No Torch import, model download or training has occurred in this cell.")


## Prepare the documented GPU environment

The next cell installs only missing/mismatching pinned dependencies using this notebook's Python. On first use this can download the CUDA PyTorch wheel; model weights are downloaded later only after a valid CUDA/BF16 check. On a Mac it stops before installation. If an incompatible runtime was already imported into this kernel, restart the kernel after correcting the environment rather than running with mixed versions.


In [ ]:
import importlib.metadata
if platform.system() == "Darwin":
    raise RuntimeError("GPU training preparation stops on the Mac. Use scripts/train_bert.py --preflight for CPU-only input checks; run this notebook on the GPU machine.")
expected = {"torch":"2.9.0", "transformers":"4.51.3", "huggingface-hub":"0.36.2", "numpy":"1.26.4", "pandas":"2.2.3", "scikit-learn":"1.6.1", "safetensors":"0.5.3"}
def installed(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None
torch_version = installed("torch")
if torch_version != "2.9.0+cu128":
    if "torch" in sys.modules:
        raise RuntimeError("This kernel already imported an incompatible Torch build. Install torch 2.9.0 cu128 in the documented environment and restart its kernel.")
    subprocess.check_call([sys.executable,"-m","pip","install","torch==2.9.0+cu128","--index-url","https://download.pytorch.org/whl/cu128"])
if installed("torch") != "2.9.0+cu128":
    raise RuntimeError("The CUDA wheel was not installed into this kernel environment; correct the environment and restart before training.")
needs = [name for name,version in expected.items() if name != "torch" and installed(name) != version]
if needs:
    imported = [name for name in ["numpy","pandas","transformers","sklearn"] if name in sys.modules]
    if imported:
        raise RuntimeError("Restart in the correctly installed pinned environment; dependencies are already imported: " + str(imported))
    subprocess.check_call([sys.executable,"-m","pip","install","-r",str(PACKAGE_ROOT/"requirements_gpu.txt")])
import torch
if not torch.cuda.is_available():
    raise RuntimeError("CUDA unavailable. Install the documented CUDA-enabled Torch wheel; no CPU or MPS training fallback is permitted.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("BF16 unsupported; return the diagnostic without changing the fixed recipe.")
print("Torch:",torch.__version__,"CUDA:",torch.version.cuda,"GPU:",torch.cuda.get_device_name(0))
print("Execution guards will verify GPU architecture, memory, disk space and original pretrained SHA before loading/training.")


In [ ]:
# Hardware/disk guards before the original pretrained model is downloaded.
from train_bert import check_cuda_build
import shutil
cuda_build = check_cuda_build(torch.cuda.get_device_capability(0), torch.version.cuda, torch.cuda.get_arch_list())
gpu_memory_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
if gpu_memory_gib < CHECKED["config"]["minimum_gpu_memory_gib"]:
    raise RuntimeError("Insufficient GPU memory for the fixed 16GB recipe; do not lower batch size or change settings silently.")
if shutil.disk_usage(PACKAGE_ROOT).free < 12*1024**3:
    raise RuntimeError("At least 12GiB free disk is required after dependency installation for complete checkpoints and result export.")
print(json.dumps({"cuda_build":cuda_build,"gpu_memory_gib":gpu_memory_gib,"free_disk_gib":shutil.disk_usage(PACKAGE_ROOT).free/1024**3},indent=2))
print("Hardware preflight passed. No pretrained weights loaded and no model training yet.")


## Execute the one new run

The script creates a fresh timestamped run directory and a `STARTED` marker, saves every strict improvement in six-class validation Macro F1, and uses a separate meaningful-improvement stopping clock. Original datasets and old results are not overwritten. An incomplete/interrupted started run blocks automatic reruns; retain its real partial checkpoint and logs for diagnosis. A completed run is hash-checked and reused rather than trained again.

Full output is streamed below and also saved under `outputs/`. Do not change the seed, split, weights, epoch-selection rule or tolerances to chase 0.581140. No held-out evaluation runs automatically.


In [ ]:
child_env = dict(os.environ)
child_env["PYTHONHASHSEED"] = str(CHECKED["config"]["seed"])
child_env.pop("PYTHONOPTIMIZE", None)
child_env["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
subprocess.run([sys.executable, str(PACKAGE_ROOT/"scripts/train_bert.py"), "--package-root", str(PACKAGE_ROOT), "--execute"],
               cwd=PACKAGE_ROOT, env=child_env, check=True)
completion_path = PACKAGE_ROOT/"outputs/COMPLETED.json"
if not completion_path.is_file():
    raise RuntimeError("No completed run marker exists. Preserve logs/partial artifacts and return the actual failure; do not claim training or export success.")
completion = json.loads(completion_path.read_text(encoding="utf-8"))
if completion.get("status") != "COMPLETED_NEW_VALIDATION_ONLY_RUN":
    raise RuntimeError("Final run status is not complete")
print("Selected new epoch:",completion["selected_epoch"])
print("Actual new validation Macro F1:",completion["selected_validation_macro_f1"])
print("Complete result ZIP:",PACKAGE_ROOT/completion["return_zip"])
print("Completion proof:",completion_path)


## Return the actual artifacts

Save this executed notebook (**Ctrl+S**). Return **both** the complete result ZIP and `outputs/COMPLETED.json`, plus this executed notebook. The ZIP contains the full selected `model.safetensors`, config/tokenizer, optimizer/RNG state, class mapping/weights, validation predictions/probabilities/logits, logs, environment, checkpoint replay and independent recalculated metrics. It is not a reports-only archive. Original packaged data/cache and any held-out data are excluded from the result ZIP.

The completion JSON binds the final ZIP SHA-256 and every finalized run artifact and is published only after independent checks and ZIP verification pass. This avoids claiming completion when a checkpoint/export is missing. If execution failed, return the failed run's actual status/logs and last complete candidate checkpoint instead; do not fabricate a passing marker or restart another trial automatically.

Freeze and independently inspect the returned artifacts before separately authorizing one final held-out evaluation. This notebook supplies no such evaluation command.


In [ ]:
from IPython.display import display, FileLink
if completion_path.is_file():
    # Notebook is distributed in notebooks/, so browser links ascend to package root.
    print("Return the two files below plus this saved executed notebook:")
    display(FileLink("../" + completion["return_zip"]))
    display(FileLink("../outputs/COMPLETED.json"))
